In [ ]:
# imports and Drive mount
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split

drive.mount("/content/drive", force_remount=True)
PROJECT = Path("/content/drive/MyDrive/asr project")
MCI_AUDIO_ROOT = PROJECT / "preprocessed_patient_audio"
CONTROL_AUDIO_ROOT = PROJECT / "preprocessed_patient_audio/Control"
SPLIT_ROOT = PROJECT / "splits"
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert MCI_AUDIO_ROOT.exists(), f"MCI audio folder not found: {MCI_AUDIO_ROOT}"
assert CONTROL_AUDIO_ROOT.exists(), f"Control audio folder not found: {CONTROL_AUDIO_ROOT}"

print("Project:", PROJECT)
print("MCI root:", MCI_AUDIO_ROOT)
print("Control root:", CONTROL_AUDIO_ROOT)
print("Split output:", SPLIT_ROOT)

In [ ]:
#define folders
MCI_AUDIO_ROOT = PROJECT / "preprocessed_patient_audio"
CONTROL_AUDIO_ROOT = PROJECT / "preprocessed_patient_audio/Control"

In [ ]:
def infer_language_from_path(path):
    """
    Infer language from path.
    """

    # convert the path to searchable text
    path_str = str(path)

    # identify english dataset paths
    if "Pitt" in path_str or "Pits" in path_str or "English" in path_str:
        return "English"

    # identify greek dataset paths
    if "Greek" in path_str or "DemCare" in path_str or "Dem@Care" in path_str:
        return "Greek"

    # identify mandarin dataset paths
    if "Mandarin" in path_str or "Chou" in path_str:
        return "Mandarin"

    return "Unknown"


def load_wav_mono_16k_np(wav_path, target_sr=16000):
    """
    Load patient-only WAV as mono 16 kHz numpy array.
    Use only if you need waveform data.
    """

    # load resample and convert the audio to mono
    x, sr = librosa.load(
        str(wav_path),
        sr=target_sr,
        mono=True
    )

    # convert the waveform to float32
    x = x.astype(np.float32)

    # find the maximum absolute amplitude
    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0

    # normalize non silent audio
    if max_abs > 0:
        x = x / max_abs

    # return the waveform and target sample rate
    return x, target_sr


def collect_patient_wavs(root, label_name, label_value):
    """
    Collect patient-only WAV paths.

    Critical fix:
        - For MCI, exclude anything inside /Control/.
        - For Control, collect only from CONTROL_AUDIO_ROOT.
    """
    root = Path(root)

    # find all patient-only recordings
    wavs = sorted(root.rglob("*_patient.wav"))
    rows = []

    # inspect each patient-only recording
    for wav_path in wavs:
        path_str = str(wav_path)

        # exclude control recordings from the mci group
        if label_name == "MCI" and "/Control/" in path_str:
            continue

        # keep control recordings only from control folders
        if label_name == "Control" and "/Control/" not in path_str:
            continue

        # extract recording metadata
        dataset = wav_path.parents[1].name
        file_id = wav_path.stem.replace("_patient", "")
        language = infer_language_from_path(wav_path)

        # add one metadata row for the recording
        rows.append({
            "dataset": dataset,
            "language": language,
            "file_id": file_id,
            "label_name": label_name,
            "label": label_value,
            "wav_path": str(wav_path),
        })

    # return the collected recordings as a dataframe
    return pd.DataFrame(rows)


In [ ]:
def make_safe_speaker_id(row):
    """
    Create safe speaker IDs.

    English/Pitt:
        Use full file_id as speaker_id.

    Greek:
        Use full file_id as speaker_id.

    Mandarin:
        006_Daddy, 006_market, 006_park -> base_id 006.
        Include label_name because Control 006 and MCI 006 may both exist.
    """

    # read language label and recording identifiers
    language = str(row["language"])
    label_name = str(row["label_name"])
    file_id = str(row["file_id"])

    # group repeated english recordings by participant
    if language == "English":
        # extract the participant id before the hyphen
        if "-" in file_id:
            base_id = file_id.split("-")[0]
        else:
            base_id = file_id

        # include language and diagnostic group
        return f"{language}_{label_name}_{base_id}"

    # use the full greek recording id
    if language == "Greek":
        return f"{language}_{label_name}_{file_id}"

    # group repeated mandarin recordings by participant
    if language == "Mandarin":
        # extract the participant id before the separator
        if "_" in file_id:
            base_id = file_id.split("_")[0]
        elif "-" in file_id:
            base_id = file_id.split("-")[0]
        else:
            base_id = file_id

        # include language and diagnostic group
        return f"{language}_{label_name}_{base_id}"

    # use the full id for unknown languages
    return f"{language}_{label_name}_{file_id}"


def add_safe_ids(df):
    """
    Add speaker_id and unique_audio_id.
    """

    # copy the dataframe before adding identifiers
    df = df.copy()

    # create one safe speaker id per row
    df["speaker_id"] = df.apply(make_safe_speaker_id, axis=1)

    # create one unique identifier per recording
    df["unique_audio_id"] = (
        df["language"].astype(str)
        + "||"
        + df["label_name"].astype(str)
        + "||"
        + df["speaker_id"].astype(str)
        + "||"
        + df["file_id"].astype(str)
    )
    return df


In [ ]:

#  collect MCI and Control patient-only WAVs

# collect recordings assigned to the mci class
mci_df = collect_patient_wavs(
    root=MCI_AUDIO_ROOT,
    label_name="MCI",
    label_value=1
)

# collect recordings assigned to the control class
control_df = collect_patient_wavs(
    root=CONTROL_AUDIO_ROOT,
    label_name="Control",
    label_value=0
)

# display the number of collected recordings
print("MCI collected:", mci_df.shape)
print("Control collected:", control_df.shape)

# confirm no control paths were included as mci
print("\nMCI paths containing /Control/ should be 0:")
print(mci_df["wav_path"].str.contains("/Control/").sum())

# confirm all control paths come from control folders
print("\nControl paths not containing /Control/ should be 0:")
print((~control_df["wav_path"].str.contains("/Control/")).sum())

# combine both diagnostic groups
patient_audio_df = pd.concat(
    [mci_df, control_df],
    ignore_index=True
)

# add speaker-safe and recording-safe identifiers
patient_audio_df = add_safe_ids(patient_audio_df)

# show the dataset size before duplicate removal
print("\nBefore duplicate cleanup:", patient_audio_df.shape)

# inspect class counts for each language
print("\nCounts by language and label:")
print(patient_audio_df.groupby(["language", "label_name"]).size())

patient_audio_df.head()


In [ ]:
#  inspect duplicate logical rows

# find repeated logical recording entries
duplicate_mask = patient_audio_df.duplicated(
    subset=["language", "label_name", "speaker_id", "file_id"],
    keep=False
)

# keep all rows involved in duplication
duplicates_df = patient_audio_df[duplicate_mask].copy()

# show the number of duplicate rows
print("Duplicate logical rows found:", len(duplicates_df))

# display duplicate details when any are found
if len(duplicates_df) > 0:

    # allow longer file paths in the output
    pd.set_option("display.max_colwidth", 300)

    # show the first 100 duplicate rows
    display(
        duplicates_df[
            [
                "dataset",
                "language",
                "file_id",
                "label_name",
                "speaker_id",
                "unique_audio_id",
                "wav_path",
            ]
        ].sort_values(
            ["language", "label_name", "speaker_id", "file_id", "wav_path"]
        ).head(100)
    )


In [ ]:
#remove dumplicates

# keep only the first copy of each logical recording
patient_audio_df = patient_audio_df.drop_duplicates(
    subset=["language", "label_name", "speaker_id", "file_id"],
    keep="first"
).reset_index(drop=True)

# show the dataset size after duplicate removal
print("After duplicate cleanup:", patient_audio_df.shape)

# confirm every audio identifier is unique
print("\nDuplicated unique_audio_id:")
print(patient_audio_df["unique_audio_id"].duplicated().sum())

# show class counts after cleanup
print("\nCounts by language and label after cleanup:")
print(patient_audio_df.groupby(["language", "label_name"]).size())

In [ ]:
# find mci rows linked to control folders
bad_mci_paths = patient_audio_df[
    (patient_audio_df["label_name"] == "MCI") &
    (patient_audio_df["wav_path"].str.contains("/Control/"))
]

# show the number of incorrect mci paths
print("MCI rows with Control path:", len(bad_mci_paths))

# display incorrect rows when any are found
if len(bad_mci_paths) > 0:
    display(
        bad_mci_paths[
            ["file_id", "label_name", "speaker_id", "wav_path"]
        ].head(50)
    )

assert len(bad_mci_paths) == 0, "MCI rows are still pointing to Control paths!"

In [ ]:
# Mandarin sanity check

# keep only mandarin recordings
mandarin_df = patient_audio_df[
    patient_audio_df["language"] == "Mandarin"
].copy()

# extract the participant id before the first underscore
mandarin_df["mandarin_base_id"] = (
    mandarin_df["file_id"]
    .astype(str)
    .str.split("_")
    .str[0]
)

print("Mandarin base IDs that appear in both MCI and Control:")

# count the number of labels linked to each base id
base_id_check = (
    mandarin_df
    .groupby("mandarin_base_id")["label_name"]
    .nunique()
    .reset_index(name="n_labels")
)

# keep base ids appearing in more than one label
base_ids_in_both = base_id_check[
    base_id_check["n_labels"] > 1
]


print("Count:", len(base_ids_in_both))
display(base_ids_in_both.head(20))

print("\nThis is okay because speaker_id includes label_name.")

# count labels linked to each safe speaker id
safe_speaker_check = (
    mandarin_df
    .groupby("speaker_id")["label_name"]
    .nunique()
    .reset_index(name="n_labels")
)

# find safe speaker ids crossing diagnostic groups
bad_safe_speakers = safe_speaker_check[
    safe_speaker_check["n_labels"] > 1
]

assert len(bad_safe_speakers) == 0, "Problem: safe Mandarin speaker_id crosses labels."


In [ ]:
# inspect Mandarin 006 if present
if "006" in mandarin_df["mandarin_base_id"].values:
    display(
        mandarin_df[
            mandarin_df["mandarin_base_id"] == "006"
        ][[
            "file_id",
            "label_name",
            "mandarin_base_id",
            "speaker_id",
            "unique_audio_id",
            "wav_path",
        ]].sort_values(["label_name", "file_id"])
    )

In [ ]:
# English / Pitt sanity check

# keep only english recordings
english_df = patient_audio_df[
    patient_audio_df["language"] == "English"
].copy()

print("English rows:", len(english_df))
print("English unique file_ids:", english_df["file_id"].nunique())
print("English unique speaker_ids:", english_df["speaker_id"].nunique())

print("\nEnglish duplicate file_id rows:")
english_dup_file = english_df[
    english_df.duplicated(subset=["file_id"], keep=False)
].copy()
print(len(english_dup_file))

# display duplicate file details when found
if len(english_dup_file) > 0:
    display(
        english_dup_file[
            ["file_id", "label_name", "speaker_id", "wav_path"]
        ].sort_values(["file_id", "label_name"]).head(50)
    )

print("\nEnglish duplicate speaker_id rows:")
print(english_df["speaker_id"].duplicated().sum())



In [ ]:
# Split into 6 variables


def split_into_6_variables(df):
    English_MCI = df[
        (df["language"] == "English") &
        (df["label_name"] == "MCI")
    ].copy()

    Greek_MCI = df[
        (df["language"] == "Greek") &
        (df["label_name"] == "MCI")
    ].copy()

    Mandarin_MCI = df[
        (df["language"] == "Mandarin") &
        (df["label_name"] == "MCI")
    ].copy()

    English_Control = df[
        (df["language"] == "English") &
        (df["label_name"] == "Control")
    ].copy()

    Greek_Control = df[
        (df["language"] == "Greek") &
        (df["label_name"] == "Control")
    ].copy()

    Mandarin_Control = df[
        (df["language"] == "Mandarin") &
        (df["label_name"] == "Control")
    ].copy()

    return (
        English_MCI,
        Greek_MCI,
        Mandarin_MCI,
        English_Control,
        Greek_Control,
        Mandarin_Control,
    )


(
    English_MCI,
    Greek_MCI,
    Mandarin_MCI,
    English_Control,
    Greek_Control,
    Mandarin_Control,
) = split_into_6_variables(patient_audio_df)

print("English_MCI:", English_MCI.shape)
print("Greek_MCI:", Greek_MCI.shape)
print("Mandarin_MCI:", Mandarin_MCI.shape)

print("English_Control:", English_Control.shape)
print("Greek_Control:", Greek_Control.shape)
print("Mandarin_Control:", Mandarin_Control.shape)

In [ ]:
# Metadata saving

CLEAN_META_ROOT = PROJECT / "clean_audio_metadata"
CLEAN_META_ROOT.mkdir(parents=True, exist_ok=True)

patient_audio_csv = CLEAN_META_ROOT / "patient_audio_metadata_clean_safe_ids.csv"

patient_audio_df.to_csv(patient_audio_csv, index=False)

print("Saved clean metadata:", patient_audio_csv)

In [ ]:
#  70/30 grouped split function

def make_grouped_language_split(
    df,
    train_size=0.70,
    random_state=42,
):
    """
    Make a 70/30 train/test split separately for each language and label.
    This prevents same speaker/task-group from appearing in both train and test.
    """

    train_parts = []
    test_parts = []
    split_info = []

    # process each language separately
    for language in sorted(df["language"].unique()):
        lang_df = df[df["language"] == language].copy()

        # split each diagnostic group separately
        for label_name in sorted(lang_df["label_name"].unique()):
            sub = lang_df[lang_df["label_name"] == label_name].copy()

            # collect unique speaker groups
            speakers = sorted(sub["speaker_id"].unique())

            # keep all data in train when fewer than two speakers exist
            if len(speakers) < 2:
                print(
                    f"Warning: not enough speakers for {language} / {label_name}. "
                    "Putting all in train."
                )

                train_parts.append(sub)

                # record the unsplit group information
                split_info.append({
                    "language": language,
                    "label_name": label_name,
                    "n_speakers_total": len(speakers),
                    "n_speakers_train": len(speakers),
                    "n_speakers_test": 0,
                    "n_files_total": len(sub),
                    "n_files_train": len(sub),
                    "n_files_test": 0,
                })

                continue

            # split complete speaker groups into train and test
            train_speakers, test_speakers = train_test_split(
                speakers,
                train_size=train_size,
                random_state=random_state,
                shuffle=True
            )

            # convert selected speaker lists to sets
            train_speakers = set(train_speakers)
            test_speakers = set(test_speakers)

            # assign all recordings from each speaker group
            sub_train = sub[sub["speaker_id"].isin(train_speakers)].copy()
            sub_test = sub[sub["speaker_id"].isin(test_speakers)].copy()

            # store the language and label subsets
            train_parts.append(sub_train)
            test_parts.append(sub_test)

            # record speaker and file counts
            split_info.append({
                "language": language,
                "label_name": label_name,
                "n_speakers_total": len(speakers),
                "n_speakers_train": len(train_speakers),
                "n_speakers_test": len(test_speakers),
                "n_files_total": len(sub),
                "n_files_train": len(sub_train),
                "n_files_test": len(sub_test),
            })

    # combine all training subsets
    train_df = pd.concat(train_parts, ignore_index=True)

    # combine test subsets when available
    if len(test_parts) > 0:
        test_df = pd.concat(test_parts, ignore_index=True)
    else:
        test_df = pd.DataFrame(columns=df.columns)

    # create the split summary table
    split_info_df = pd.DataFrame(split_info)

    # return train test and summary dataframes
    return train_df, test_df, split_info_df


In [ ]:
# create 70/30 train/test split


train_df, test_df, split_info_df = make_grouped_language_split(
    patient_audio_df,
    train_size=0.70,
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain counts:")
print(train_df.groupby(["language", "label_name"]).size())

print("\nTest counts:")
print(test_df.groupby(["language", "label_name"]).size())

print("\nSplit info:")
display(split_info_df)

In [ ]:
# leakage checks

# create unique speaker keys for the training set
train_speaker_keys = set(
    train_df["language"].astype(str)
    + "||"
    + train_df["label_name"].astype(str)
    + "||"
    + train_df["speaker_id"].astype(str)
)

# create unique speaker keys for the test set
test_speaker_keys = set(
    test_df["language"].astype(str)
    + "||"
    + test_df["label_name"].astype(str)
    + "||"
    + test_df["speaker_id"].astype(str)
)

# find speakers present in both splits
speaker_overlap = train_speaker_keys & test_speaker_keys

# display speaker overlap results
print("Speaker overlap train/test:", len(speaker_overlap))
print(sorted(speaker_overlap)[:20])

# stop when speaker leakage is found
assert len(speaker_overlap) == 0, "Speaker leakage detected."

# find recordings present in both splits
unique_overlap = set(train_df["unique_audio_id"]) & set(test_df["unique_audio_id"])
print("\nUnique audio overlap train/test:", len(unique_overlap))
print(sorted(unique_overlap)[:20])

# stop when audio leakage is found
assert len(unique_overlap) == 0, "Same audio appears in train and test."


In [ ]:
# Duplicate checks in train/test

print("Train duplicated unique_audio_id:", train_df["unique_audio_id"].duplicated().sum())
print("Test duplicated unique_audio_id:", test_df["unique_audio_id"].duplicated().sum())

print("\nTrain duplicated logical audio rows:")
print(
    train_df.duplicated(
        subset=["language", "label_name", "speaker_id", "file_id"]
    ).sum()
)

print("\nTest duplicated logical audio rows:")
print(
    test_df.duplicated(
        subset=["language", "label_name", "speaker_id", "file_id"]
    ).sum()
)

In [ ]:
# Save split CSVs
train_csv = SPLIT_ROOT / "train_70_by_language_grouped_by_safe_speaker.csv"
test_csv = SPLIT_ROOT / "test_30_by_language_grouped_by_safe_speaker.csv"
split_info_csv = SPLIT_ROOT / "split_info_70_30_by_language_safe_speaker.csv"

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)
split_info_df.to_csv(split_info_csv, index=False)

print("Saved train split:", train_csv)
print("Saved test split:", test_csv)
print("Saved split info:", split_info_csv)

In [ ]:
# Save one combined split CSV with split column
train_df_save = train_df.copy()
test_df_save = test_df.copy()

train_df_save["split"] = "train"
test_df_save["split"] = "test"

split_df = pd.concat(
    [train_df_save, test_df_save],
    ignore_index=True
)

front_cols = ["split", "dataset", "language", "file_id", "label_name", "label", "speaker_id", "unique_audio_id", "wav_path"]
front_cols = [c for c in front_cols if c in split_df.columns]
other_cols = [c for c in split_df.columns if c not in front_cols]

split_df = split_df[front_cols + other_cols]

combined_split_csv = SPLIT_ROOT / "FIN_split_70_30_by_language_grouped_by_safe_speaker.csv"
split_info_csv = SPLIT_ROOT / "FIN_split_info_70_30_by_language_safe_speaker.csv"

split_df.to_csv(combined_split_csv, index=False)
split_info_df.to_csv(split_info_csv, index=False)

print("Saved combined split:", combined_split_csv)
print("Saved split info:", split_info_csv)

print("\nSplit counts:")
print(split_df.groupby(["split", "language", "label_name"]).size())

In [ ]:
#per-language train/test dataframes

# keep english training rows
English_train_df = train_df[train_df["language"] == "English"].copy()

# keep english test rows
English_test_df = test_df[test_df["language"] == "English"].copy()

# keep greek training rows
Greek_train_df = train_df[train_df["language"] == "Greek"].copy()

# keep greek test rows
Greek_test_df = test_df[test_df["language"] == "Greek"].copy()

# keep mandarin training rows
Mandarin_train_df = train_df[train_df["language"] == "Mandarin"].copy()

# keep mandarin test rows
Mandarin_test_df = test_df[test_df["language"] == "Mandarin"].copy()

#  english split sizes
print("English train:", English_train_df.shape)
print("English test:", English_test_df.shape)

#  greek split sizes
print("Greek train:", Greek_train_df.shape)
print("Greek test:", Greek_test_df.shape)

#  mandarin split sizes
print("Mandarin train:", Mandarin_train_df.shape)
print("Mandarin test:", Mandarin_test_df.shape)


In [ ]:
# IDs for display and exact matching

# collect file ids for readable display
english_train_file_ids = English_train_df["file_id"].tolist()
english_test_file_ids = English_test_df["file_id"].tolist()

greek_train_file_ids = Greek_train_df["file_id"].tolist()
greek_test_file_ids = Greek_test_df["file_id"].tolist()

mandarin_train_file_ids = Mandarin_train_df["file_id"].tolist()
mandarin_test_file_ids = Mandarin_test_df["file_id"].tolist()

# collect unique audio ids for exact matching
english_train_unique_ids = English_train_df["unique_audio_id"].tolist()
english_test_unique_ids = English_test_df["unique_audio_id"].tolist()

greek_train_unique_ids = Greek_train_df["unique_audio_id"].tolist()
greek_test_unique_ids = Greek_test_df["unique_audio_id"].tolist()

mandarin_train_unique_ids = Mandarin_train_df["unique_audio_id"].tolist()
mandarin_test_unique_ids = Mandarin_test_df["unique_audio_id"].tolist()


print("English train file IDs:", len(english_train_file_ids))
print("English test file IDs:", len(english_test_file_ids))

print("Greek train file IDs:", len(greek_train_file_ids))
print("Greek test file IDs:", len(greek_test_file_ids))

print("Mandarin train file IDs:", len(mandarin_train_file_ids))
print("Mandarin test file IDs:", len(mandarin_test_file_ids))
print("\nUnique ID checks:")
print("English train:", len(english_train_unique_ids), len(set(english_train_unique_ids)))
print("English test:", len(english_test_unique_ids), len(set(english_test_unique_ids)))
print("Greek train:", len(greek_train_unique_ids), len(set(greek_train_unique_ids)))
print("Greek test:", len(greek_test_unique_ids), len(set(greek_test_unique_ids)))
print("Mandarin train:", len(mandarin_train_unique_ids), len(set(mandarin_train_unique_ids)))
print("Mandarin test:", len(mandarin_test_unique_ids), len(set(mandarin_test_unique_ids)))


In [ ]:
#  save ID lists


id_lists = {
    "english_train_file_ids": english_train_file_ids,
    "english_test_file_ids": english_test_file_ids,
    "greek_train_file_ids": greek_train_file_ids,
    "greek_test_file_ids": greek_test_file_ids,
    "mandarin_train_file_ids": mandarin_train_file_ids,
    "mandarin_test_file_ids": mandarin_test_file_ids,

    "english_train_unique_ids": english_train_unique_ids,
    "english_test_unique_ids": english_test_unique_ids,
    "greek_train_unique_ids": greek_train_unique_ids,
    "greek_test_unique_ids": greek_test_unique_ids,
    "mandarin_train_unique_ids": mandarin_train_unique_ids,
    "mandarin_test_unique_ids": mandarin_test_unique_ids,
}

for name, ids in id_lists.items():
    out_path = SPLIT_ROOT / f"{name}.txt"

    with open(out_path, "w") as f:
        for item in ids:
            f.write(str(item) + "\n")

    print("Saved:", out_path)

In [ ]:
# summary
print("Done.")
print("\nClean metadata:")
print(patient_audio_csv)
print("\nTrain CSV:")
print(train_csv)
print("\nTest CSV:")
print(test_csv)
print("\nSplit info:")
print(split_info_csv)
print("\nUse train_df and test_df for modeling.")
print("Use unique_audio_id for exact audio matching.")
print("Use speaker_id for speaker-level leakage checks.")
print("Use file_id only for display.")